# AI Agent Workforce Orchestrator

A local multi-agent workflow prototype demonstrating task decomposition, specialist agents, dependency-aware scheduling, shared state, tool allowlists, quality review, bounded retries, and execution tracing.

The demo uses deterministic Python agents and synthetic feedback; no API key or external service is required.

## 1. Install dependencies

In [ ]:
%pip install -q pandas matplotlib

## 2. Imports and workflow structures

In [ ]:

from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Callable, Any
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

@dataclass
class Task:
    task_id: str
    title: str
    agent_name: str
    description: str
    dependencies: list[str] = field(default_factory=list)
    status: str = "pending"
    attempts: int = 0
    output: Any = None
    error: str | None = None

@dataclass
class Agent:
    name: str
    role: str
    execute_fn: Callable
    def execute(self, task, context, tools):
        return self.execute_fn(task, context, tools)

class Workflow:
    def __init__(self, name, tasks):
        self.name = name
        self.tasks = {t.task_id: t for t in tasks}
        self.trace = []
    def log(self, task, event, detail=""):
        self.trace.append({
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "task_id": task.task_id, "agent": task.agent_name,
            "event": event, "detail": str(detail)[:500]
        })


## 3. Safe demo tools

Tools are explicitly allowlisted and operate only on in-memory synthetic data.

In [ ]:

FEEDBACK = [
    {"id": 1, "text": "I waited too long for a support response.", "category": "response_time"},
    {"id": 2, "text": "The billing help article did not solve my issue.", "category": "knowledge_gap"},
    {"id": 3, "text": "The agent explained the solution clearly.", "category": "positive"},
    {"id": 4, "text": "I had to explain my issue more than once.", "category": "handoff"},
    {"id": 5, "text": "The refund status was difficult to understand.", "category": "process_clarity"},
]
def fetch_feedback():
    return FEEDBACK.copy()
def summarize_categories(rows):
    counts = {}
    for row in rows:
        counts[row["category"]] = counts.get(row["category"], 0) + 1
    return counts
TOOLS = {"fetch_feedback": fetch_feedback, "summarize_categories": summarize_categories}


## 4. Define specialist agents

In [ ]:

def planner(task, context, tools):
    return {"objective": "Improve customer support experience",
            "workstreams": ["Understand feedback", "Prioritize improvements",
                            "Create action plan", "Review evidence"],
            "success_metrics": ["first response time", "resolution time", "repeat contact rate", "CSAT"]}

def researcher(task, context, tools):
    rows = tools["fetch_feedback"]()
    return {"sample_size": len(rows), "feedback": rows,
            "theme_counts": tools["summarize_categories"](rows),
            "limitation": "Small illustrative synthetic sample."}

def analyst(task, context, tools):
    counts = context["research"]["theme_counts"]
    ranked = sorted(counts.items(), key=lambda x: x[1], reverse=True)
    return {"priorities": [{"theme": k, "mentions": v} for k, v in ranked],
            "insights": ["Investigate response delays and support coverage.",
                         "Improve billing and refund help content.",
                         "Reduce repeated explanations during handoffs."],
            "caveat": "Priorities are based on a tiny synthetic dataset."}

def writer(task, context, tools):
    a = context["analysis"]
    return {"executive_summary": "Illustrative feedback suggests opportunities in response speed, knowledge quality, and support handoffs.",
            "recommendations": [f"Investigate {x['theme'].replace('_',' ')} and define a measurable improvement." for x in a["priorities"]],
            "measurement_plan": ["Track median first response time weekly.",
                                 "Track resolution time and repeat contact rate.",
                                 "Review CSAT alongside operational metrics."],
            "next_steps": ["Validate themes with representative feedback.",
                           "Assign owners and metric baselines.",
                           "Run a time-boxed pilot and compare results."]}

def reviewer(task, context, tools):
    draft = context["draft"]
    required = ["executive_summary", "recommendations", "measurement_plan", "next_steps"]
    missing = [k for k in required if not draft.get(k)]
    return {"approved": not missing, "missing_sections": missing,
            "review_notes": "Evidence limitations are stated." if not missing else "Complete missing sections."}

AGENTS = {
    "Workforce Planner": Agent("Workforce Planner", "Decomposes objectives", planner),
    "Feedback Researcher": Agent("Feedback Researcher", "Collects evidence", researcher),
    "Business Analyst": Agent("Business Analyst", "Prioritizes findings", analyst),
    "Report Writer": Agent("Report Writer", "Creates business brief", writer),
    "Quality Reviewer": Agent("Quality Reviewer", "Reviews deliverable", reviewer),
}


## 5. Define the task graph

In [ ]:

tasks = [
    Task("plan", "Decompose business objective", "Workforce Planner", "Define workstreams."),
    Task("research", "Collect feedback", "Feedback Researcher", "Retrieve and group sample feedback.", ["plan"]),
    Task("analysis", "Prioritize opportunities", "Business Analyst", "Interpret themes.", ["research"]),
    Task("draft", "Write improvement brief", "Report Writer", "Create recommendations.", ["analysis"]),
    Task("review", "Review final brief", "Quality Reviewer", "Check completeness.", ["draft"]),
]
workflow = Workflow("Customer Support Improvement Plan", tasks)

def validate_graph(wf):
    for t in wf.tasks.values():
        for dep in t.dependencies:
            if dep not in wf.tasks:
                raise ValueError(f"Unknown dependency {dep} in {t.task_id}")
    indegree = {k: len(t.dependencies) for k, t in wf.tasks.items()}
    queue = [k for k, v in indegree.items() if v == 0]
    visited = 0
    while queue:
        node = queue.pop(0); visited += 1
        for key, task in wf.tasks.items():
            if node in task.dependencies:
                indegree[key] -= 1
                if indegree[key] == 0: queue.append(key)
    if visited != len(wf.tasks):
        raise ValueError("Dependency cycle detected")

def make_context(wf):
    done = {k: t.output for k, t in wf.tasks.items() if t.status == "completed"}
    return {name: done[key] for key, name in [
        ("plan","plan"), ("research","research"),
        ("analysis","analysis"), ("draft","draft")
    ] if key in done}

def run_workflow(wf, agents, tools, max_retries=1):
    validate_graph(wf)
    pending = set(wf.tasks)
    while pending:
        ready = [wf.tasks[k] for k in sorted(pending)
                 if all(wf.tasks[d].status == "completed" for d in wf.tasks[k].dependencies)]
        if not ready:
            raise RuntimeError(f"Blocked tasks: {pending}")
        for task in ready:
            agent = agents[task.agent_name]
            while task.attempts <= max_retries:
                task.attempts += 1
                task.status = "running"
                wf.log(task, "started", f"Attempt {task.attempts}")
                try:
                    task.output = agent.execute(task, make_context(wf), tools)
                    task.status = "completed"
                    wf.log(task, "completed", "Structured output created")
                    break
                except Exception as exc:
                    task.error = repr(exc)
                    wf.log(task, "error", task.error)
                    if task.attempts > max_retries:
                        task.status = "failed"
            pending.remove(task.task_id)
    return wf


## 6. Execute the workforce

In [ ]:

workflow = run_workflow(workflow, AGENTS, TOOLS, max_retries=1)
status_df = pd.DataFrame([{
    "task_id": t.task_id, "title": t.title, "agent": t.agent_name,
    "status": t.status, "attempts": t.attempts, "error": t.error
} for t in workflow.tasks.values()])
display(status_df)


## 7. Inspect the deliverable and reviewer gate

In [ ]:

brief = workflow.tasks["draft"].output
review = workflow.tasks["review"].output
print("Review:", json.dumps(review, indent=2))
print("\nEXECUTIVE SUMMARY\n", brief["executive_summary"])
print("\nRECOMMENDATIONS\n- " + "\n- ".join(brief["recommendations"]))
print("\nMEASUREMENT PLAN\n- " + "\n- ".join(brief["measurement_plan"]))
print("\nNEXT STEPS\n- " + "\n- ".join(brief["next_steps"]))


## 8. Observability and workflow metrics

In [ ]:

trace_df = pd.DataFrame(workflow.trace)
display(trace_df)
status_df["status"].value_counts().plot(kind="bar", figsize=(6,4), title="Task status")
plt.xlabel("Status"); plt.ylabel("Tasks"); plt.xticks(rotation=0); plt.grid(axis="y", alpha=.25); plt.show()


## 9. Export artifacts

In [ ]:

out = Path("outputs")
out.mkdir(exist_ok=True)
(out / "business_brief.json").write_text(json.dumps({"brief": brief, "review": review}, indent=2), encoding="utf-8")
status_df.to_csv(out / "task_status.csv", index=False)
trace_df.to_csv(out / "workflow_trace.csv", index=False)
summary = {"workflow": workflow.name, "tasks": len(workflow.tasks),
           "completed": sum(t.status == "completed" for t in workflow.tasks.values()),
           "failed": sum(t.status == "failed" for t in workflow.tasks.values()),
           "review_approved": review["approved"]}
(out / "workflow_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Saved:", out.resolve())


## 10. Production roadmap and safety

Potential extensions:
1. Replace deterministic functions with an LLM adapter and validated structured outputs.
2. Add LangGraph/CrewAI, agent-to-agent messaging, and asynchronous execution.
3. Add durable state, resumable jobs, vector memory, and human feedback.
4. Add per-agent tool permissions, sandboxing, scoped credentials, and approval gates.
5. Add token budgets, timeouts, rate limits, concurrency controls, and evaluation suites.
6. Build a FastAPI backend and React/Streamlit monitoring dashboard.

**Safety:** Never grant agents unrestricted shell, browser, database, email, or payment access. Require human approval for external side effects and consequential actions. Keep secrets out of notebooks and logs. The included demo uses synthetic in-memory data and does not connect to external services.